# Etapa 2 — régua de validação e baselines
Antes de comparar modelos, montamos a régua que vai julgar todos eles: semanas de validação separadas no tempo e a métrica da competição (MAP@12). Depois medimos baselines simples, que definem a nota mínima que qualquer modelo precisa superar.

In [1]:
# Bibliotecas para trabalhar com os dados
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.dataset as ds

# Gráficos
import plotly.express as px
import plotly.io as pio

# Sistema
from datetime import date
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

pio.templates.default = 'plotly_dark'
px.defaults.height = 500

## 1. Dados de 2020
### Recorte das transações
A competição pede a semana seguinte a **22/09/2020**, então trabalhamos com o fim do histórico, não com o começo (2018) usado na etapa 1. O CSV completo tem 3,5 GB: lemos uma única vez só as linhas a partir de `DATA_INICIO` e salvamos em parquet, que abre em segundos nas próximas execuções.

In [2]:
PASTA_DADOS = Path(".")
if not (PASTA_DADOS / "transactions_train.csv").is_file():
    PASTA_DADOS = Path("H&M - Hackaton")

DATA_INICIO = date(2020, 6, 1)
ARQUIVO_PARQUET = PASTA_DADOS / "transactions_2020.parquet"


In [3]:
if not ARQUIVO_PARQUET.is_file():
    formato_csv = ds.CsvFileFormat(
        convert_options=pv.ConvertOptions(column_types={
              't_dat': pa.date32()
            , 'customer_id': pa.string()
            , 'article_id': pa.string()
        })
    )

    tabela = ds.dataset(PASTA_DADOS / "transactions_train.csv", format=formato_csv).to_table(
        filter=ds.field('t_dat') >= DATA_INICIO
    )

    tabela.to_pandas(date_as_object=False).to_parquet(ARQUIVO_PARQUET, index=False)

In [4]:
df_transaction = pd.read_parquet(ARQUIVO_PARQUET)

df_transaction['customer_id'] = df_transaction['customer_id'].astype('string[pyarrow]')
df_transaction['article_id'] = df_transaction['article_id'].astype('string[pyarrow]')

df_transaction

,t_dat,customer_id,article_id,price,sales_channel_id
0,2020-06-01,00075ef36696a7b4ed8c83e22a4bf7ea7c90ee110991ec...,0844198001,0.016932,2
1,2020-06-01,000b31552d3785c79833262bbeefa484cbc43d7b612b3c...,0777016001,0.030492,1
...,...,...,...,...,...
5151468,2020-09-22,fff4d3a8b1f3b60af93e78c30a7cb4cf75edaf2590d3e5...,0833459002,0.006763,1
5151469,2020-09-22,fffef3b6b73545df065b521e19f64bf6fe93bfd450ab20...,0898573003,0.033881,2


In [5]:
print('Período:', df_transaction['t_dat'].min().date(), 'a', df_transaction['t_dat'].max().date())
print('Itens vendidos:', len(df_transaction))
print('Clientes:', df_transaction['customer_id'].nunique())
print('Artigos:', df_transaction['article_id'].nunique())

Período: 2020-06-01 a 2020-09-22
Itens vendidos: 5151470


Clientes: 599002
Artigos: 45975


### Idade dos clientes
Usada por um dos baselines. Faixas etárias como texto; quem não tem idade no cadastro fica sem faixa.

In [6]:
df_clientes = pd.read_csv(
    PASTA_DADOS / "customers.csv",
    usecols=['customer_id', 'age'],
    dtype={"customer_id": "string[pyarrow]"},
)

df_clientes['faixa_idade'] = pd.cut(
      df_clientes['age']
    , bins=[0, 24, 34, 44, 54, 64, 120]
    , labels=['até 24', '25-34', '35-44', '45-54', '55-64', '65+']
).astype('string')

display(df_clientes['faixa_idade'].value_counts(dropna=False))

faixa_idade
25-34     393292
até 24    357169
           ...  
65+        43602
<NA>       15861
Name: count, Length: 7, dtype: Int64

## 2. A régua: semanas de validação
### Como o tempo é dividido
Cada **semana avaliada** é um "ensaio" da competição: o baseline (ou modelo) só enxerga o histórico **antes** dela e tenta adivinhar as compras dentro dela. Usamos três semanas de validação (A, B, C) para ver se um resultado se repete, e não foi sorte de uma semana só.

A última semana do arquivo fica **reservada como teste**: só vamos abri-la uma vez, no fim do projeto, com o modelo já escolhido. Nada de `train_test_split` aleatório aqui: misturar dias do futuro no treino deixa a nota otimista.

In [7]:
CORTE_A = pd.Timestamp('2020-08-26')
CORTE_B = pd.Timestamp('2020-09-02')
CORTE_C = pd.Timestamp('2020-09-09')
CORTE_TESTE = pd.Timestamp('2020-09-16')


def separar_semana(df, data_corte, dias=7):
    historico = df.loc[df['t_dat'] < data_corte]
    semana = df.loc[
          (df['t_dat'] >= data_corte)
        & (df['t_dat'] < data_corte + pd.Timedelta(days=dias))
    ]
    return historico, semana


historico_a, semana_a = separar_semana(df_transaction, CORTE_A)
historico_b, semana_b = separar_semana(df_transaction, CORTE_B)
historico_c, semana_c = separar_semana(df_transaction, CORTE_C)

In [8]:
inicio = df_transaction['t_dat'].min()
uma_semana = pd.Timedelta(days=7)

linha_do_tempo = pd.DataFrame([
      {'ensaio': 'Semana A', 'parte': 'Histórico visível', 'inicio': inicio, 'fim': CORTE_A}
    , {'ensaio': 'Semana A', 'parte': 'Semana avaliada', 'inicio': CORTE_A, 'fim': CORTE_A + uma_semana}
    , {'ensaio': 'Semana B', 'parte': 'Histórico visível', 'inicio': inicio, 'fim': CORTE_B}
    , {'ensaio': 'Semana B', 'parte': 'Semana avaliada', 'inicio': CORTE_B, 'fim': CORTE_B + uma_semana}
    , {'ensaio': 'Semana C', 'parte': 'Histórico visível', 'inicio': inicio, 'fim': CORTE_C}
    , {'ensaio': 'Semana C', 'parte': 'Semana avaliada', 'inicio': CORTE_C, 'fim': CORTE_C + uma_semana}
    , {'ensaio': 'Teste final', 'parte': 'Histórico visível', 'inicio': inicio, 'fim': CORTE_TESTE}
    , {'ensaio': 'Teste final', 'parte': 'Reservado (abrir só no fim)', 'inicio': CORTE_TESTE, 'fim': CORTE_TESTE + uma_semana}
])

fig = px.timeline(
      linha_do_tempo
    , x_start='inicio'
    , x_end='fim'
    , y='ensaio'
    , color='parte'
    , color_discrete_map={
          'Histórico visível': '#3274A1'
        , 'Semana avaliada': '#C97932'
        , 'Reservado (abrir só no fim)': '#8C8C8C'
    }
    , title='Validação temporal: cada ensaio só enxerga o passado'
)

fig.update_yaxes(autorange='reversed', title='')
fig.update_layout(height=350, legend_title='')
fig.show()

In [9]:
compras_dia = df_transaction.groupby('t_dat').size().reset_index(name='itens')

fig = px.line(
      compras_dia
    , x='t_dat'
    , y='itens'
    , title='Itens vendidos por dia e semanas usadas na validação'
)

fig.add_vrect(x0=CORTE_A, x1=CORTE_A + uma_semana, fillcolor='#C97932', opacity=0.25, line_width=0, annotation_text='A')
fig.add_vrect(x0=CORTE_B, x1=CORTE_B + uma_semana, fillcolor='#C97932', opacity=0.25, line_width=0, annotation_text='B')
fig.add_vrect(x0=CORTE_C, x1=CORTE_C + uma_semana, fillcolor='#C97932', opacity=0.25, line_width=0, annotation_text='C')
fig.add_vrect(x0=CORTE_TESTE, x1=CORTE_TESTE + uma_semana, fillcolor='#8C8C8C', opacity=0.35, line_width=0, annotation_text='Teste')
fig.update_yaxes(title='Itens vendidos')
fig.update_xaxes(title='')
fig.show()

In [10]:
resumo_semanas = pd.DataFrame({
      'itens_historico': [len(historico_a), len(historico_b), len(historico_c)]
    , 'itens_semana': [len(semana_a), len(semana_b), len(semana_c)]
    , 'clientes_semana': [semana_a['customer_id'].nunique(), semana_b['customer_id'].nunique(), semana_c['customer_id'].nunique()]
}, index=['Semana A', 'Semana B', 'Semana C'])

display(resumo_semanas)

,itens_historico,itens_semana,clientes_semana
Semana A,4108080,283181,80253
Semana B,4391261,264657,75822
Semana C,4655918,255241,72019


### A métrica: MAP@12
Para cada cliente que comprou na semana, entregamos **12 artigos em ordem**. O *average precision* (AP) olha cada acerto e pergunta: *até essa posição, que fração da lista estava certa?* Depois tira a média desses valores, dividindo por `min(artigos comprados, 12)`.

Exemplo: o cliente comprou **A** e **B**. Recomendamos `[A, X, B, Y, ...]`.
- acerto na posição 1 → precisão 1/1 = 1,00
- acerto na posição 3 → precisão 2/3 = 0,67
- AP = (1,00 + 0,67) / 2 = **0,83**

Acertar no topo da lista vale mais do que acertar no fim. O **MAP@12** é a média do AP entre os clientes; quem não recebeu nenhum acerto entra com zero.

As recomendações ficam em formato longo: uma linha por cliente × artigo, com a `posicao` de 1 a 12. Assim o cálculo vira `merge` + `groupby`, sem percorrer cliente por cliente.

In [11]:
def ap_por_cliente(recomendacoes, semana):
    compras = semana[['customer_id', 'article_id']].drop_duplicates()
    qtd_compras = compras.groupby('customer_id').size()

    acertos = (
        recomendacoes
        .merge(compras, on=['customer_id', 'article_id'], how='inner')
        .sort_values(['customer_id', 'posicao'])
    )
    acertos['acertos_ate_aqui'] = acertos.groupby('customer_id').cumcount() + 1
    acertos['precisao'] = acertos['acertos_ate_aqui'] / acertos['posicao']

    soma_precisao = acertos.groupby('customer_id')['precisao'].sum()

    # Cliente sem nenhum acerto entra com AP zero.
    soma_precisao = soma_precisao.reindex(qtd_compras.index, fill_value=0)

    return soma_precisao / qtd_compras.clip(upper=12)


def map_at_12(recomendacoes, semana):
    return ap_por_cliente(recomendacoes, semana).mean()

Conferindo a função com o exemplo acima (esperado: 0,83) e com um segundo cliente que não recebeu nenhum acerto (esperado: 0).

In [12]:
exemplo_compras = pd.DataFrame({
      'customer_id': ['cliente_1', 'cliente_1', 'cliente_2']
    , 'article_id': ['A', 'B', 'C']
})

exemplo_recomendacoes = pd.DataFrame({
      'customer_id': ['cliente_1'] * 4 + ['cliente_2'] * 2
    , 'article_id': ['A', 'X', 'B', 'Y', 'X', 'Y']
    , 'posicao': [1, 2, 3, 4, 1, 2]
})

display(ap_por_cliente(exemplo_recomendacoes, exemplo_compras).round(3))
print('MAP@12 do exemplo:', round(map_at_12(exemplo_recomendacoes, exemplo_compras), 3))

customer_id
cliente_1    0.833
cliente_2    0.000
dtype: float64

MAP@12 do exemplo: 0.417


## 3. Baselines
Regras simples, sem aprendizado. Elas definem a nota mínima: um modelo que não supera a popularidade da última semana não se justifica.

| Baseline | Regra |
| --- | --- |
| Popularidade geral | os 12 artigos mais vendidos em todo o histórico visível |
| Popularidade da última semana | os 12 mais vendidos nos 7 dias antes do corte |
| Recompra recente | os últimos artigos que o próprio cliente comprou, completados com a última semana |
| Popular na faixa etária | os 12 mais vendidos na última semana entre clientes da mesma faixa, completados com a última semana |

Todo baseline recebe o **histórico** e a lista de **clientes** a recomendar, e devolve a tabela longa `customer_id × article_id × posicao`. Ele nunca vê as compras da semana avaliada.

### Funções de apoio

In [13]:
def mais_vendidos(df, n=12):
    top = (
        df['article_id']
        .value_counts()
        .head(n)
        .rename_axis('article_id')
        .reset_index(name='itens')
    )
    top['posicao'] = top.index + 1

    return top[['article_id', 'posicao']]


def ultima_semana(historico):
    return historico.loc[historico['t_dat'] >= historico['t_dat'].max() - pd.Timedelta(days=6)]


def mesma_lista_para_todos(clientes, artigos):
    return pd.DataFrame({'customer_id': clientes}).merge(artigos, how='cross')


def completar_com(recomendacoes, reserva, clientes):
    # A reserva entra depois das recomendações próprias; repetidos saem e a lista é cortada em 12.
    reserva_todos = mesma_lista_para_todos(clientes, reserva)
    reserva_todos['posicao'] = reserva_todos['posicao'] + 100

    lista = (
        pd.concat([recomendacoes, reserva_todos])
        .sort_values(['customer_id', 'posicao'])
        .drop_duplicates(['customer_id', 'article_id'])
    )
    lista['posicao'] = lista.groupby('customer_id').cumcount() + 1

    return lista.loc[lista['posicao'] <= 12, ['customer_id', 'article_id', 'posicao']]

### As quatro regras

In [14]:
def popularidade_geral(historico, clientes):
    return mesma_lista_para_todos(clientes, mais_vendidos(historico))


def popularidade_ultima_semana(historico, clientes):
    return mesma_lista_para_todos(clientes, mais_vendidos(ultima_semana(historico)))

In [15]:
def recompra_recente(historico, clientes):
    compras_dos_clientes = historico.loc[historico['customer_id'].isin(clientes)]

    ultimas_compras = (
        compras_dos_clientes
        .groupby(['customer_id', 'article_id'])['t_dat'].max()
        .reset_index()
        .sort_values(['customer_id', 't_dat', 'article_id'], ascending=[True, False, True])
    )
    ultimas_compras['posicao'] = ultimas_compras.groupby('customer_id').cumcount() + 1

    return completar_com(
          ultimas_compras[['customer_id', 'article_id', 'posicao']]
        , mais_vendidos(ultima_semana(historico))
        , clientes
    )

In [16]:
def popular_faixa_etaria(historico, clientes):
    vendas_recentes = ultima_semana(historico).merge(df_clientes[['customer_id', 'faixa_idade']], on='customer_id', how='left')

    top_por_faixa = (
        vendas_recentes
        .groupby(['faixa_idade', 'article_id'])
        .size()
        .reset_index(name='itens')
        .sort_values(['faixa_idade', 'itens', 'article_id'], ascending=[True, False, True])
    )
    top_por_faixa['posicao'] = top_por_faixa.groupby('faixa_idade').cumcount() + 1
    top_por_faixa = top_por_faixa.loc[top_por_faixa['posicao'] <= 12]

    recomendacoes = (
        pd.DataFrame({'customer_id': clientes})
        .merge(df_clientes[['customer_id', 'faixa_idade']], on='customer_id', how='left')
        .merge(top_por_faixa[['faixa_idade', 'article_id', 'posicao']], on='faixa_idade', how='inner')
    )

    return completar_com(
          recomendacoes[['customer_id', 'article_id', 'posicao']]
        , mais_vendidos(ultima_semana(historico))
        , clientes
    )

### Como fica uma recomendação
Um cliente da semana C, com a lista de cada baseline lado a lado.

In [17]:
clientes_c = semana_c['customer_id'].unique()
CLIENTE_EXEMPLO = clientes_c[0]

exemplo = pd.DataFrame({
      'Popularidade geral': popularidade_geral(historico_c, [CLIENTE_EXEMPLO])['article_id'].values
    , 'Última semana': popularidade_ultima_semana(historico_c, [CLIENTE_EXEMPLO])['article_id'].values
    , 'Recompra recente': recompra_recente(historico_c, [CLIENTE_EXEMPLO])['article_id'].values
    , 'Faixa etária': popular_faixa_etaria(historico_c, [CLIENTE_EXEMPLO])['article_id'].values
}, index=pd.RangeIndex(1, 13, name='posicao'))

print('Cliente:', CLIENTE_EXEMPLO)
print('Comprou na semana C:', sorted(semana_c.loc[semana_c['customer_id'] == CLIENTE_EXEMPLO, 'article_id'].unique()))

with pd.option_context('display.max_rows', None):
    display(exemplo)

Cliente: 000493dd9fc463df1acc2081450c9e75ef8e87d5dd17ed6396773839f6bf71a9
Comprou na semana C: ['0399136061', '0556255001', '0576897001', '0732842014', '0732842015', '0732842021', '0816588001', '0852219003']


,Popularidade geral,Última semana,Recompra recente,Faixa etária
posicao,,,,
1,0706016001,0915526001,0733749001,0919365008
2,0372860002,0751471043,0766955004,0915526001
3,0751471001,0751471001,0872453001,0909916001
4,0599580038,0706016001,0798883008,0706016001
5,0759871002,0919365008,0831269004,0918292001
6,0610776002,0915529003,0834368002,0158340001
7,0372860001,0918292001,0862496001,0751471001
8,0817472002,0863595006,0868955003,0915529003
9,0783346001,0896152002,0886223001,0923037003


## 4. Resultado dos baselines
### MAP@12 em cada semana
Cada baseline é julgado nas três semanas, sempre recomendando para os clientes que compraram naquela semana.

In [18]:
def avaliar(baseline):
    return pd.Series({
          'Semana A': map_at_12(baseline(historico_a, semana_a['customer_id'].unique()), semana_a)
        , 'Semana B': map_at_12(baseline(historico_b, semana_b['customer_id'].unique()), semana_b)
        , 'Semana C': map_at_12(baseline(historico_c, semana_c['customer_id'].unique()), semana_c)
    })


resultados = pd.DataFrame({
      'Popularidade geral': avaliar(popularidade_geral)
    , 'Popularidade da última semana': avaliar(popularidade_ultima_semana)
    , 'Recompra recente': avaliar(recompra_recente)
    , 'Popular na faixa etária': avaliar(popular_faixa_etaria)
}).T

resultados['Média'] = resultados[['Semana A', 'Semana B', 'Semana C']].mean(axis=1)
resultados['Pior semana'] = resultados[['Semana A', 'Semana B', 'Semana C']].min(axis=1)
resultados = resultados.sort_values('Média', ascending=False)

display(resultados.round(4))

,Semana A,Semana B,Semana C,Média,Pior semana
Recompra recente,0.0194,0.0223,0.0248,0.0221,0.0194
Popular na faixa etária,0.0085,0.0091,0.0076,0.0084,0.0076
Popularidade da última semana,0.0068,0.0067,0.0066,0.0067,0.0066
Popularidade geral,0.0029,0.0041,0.0034,0.0035,0.0029


In [19]:
fig = px.bar(
      resultados[['Semana A', 'Semana B', 'Semana C']].reset_index(names='baseline').melt(id_vars='baseline', var_name='semana', value_name='MAP@12')
    , x='baseline'
    , y='MAP@12'
    , color='semana'
    , barmode='group'
    , text_auto='.4f'
    , color_discrete_sequence=['#3274A1', '#5A9BC9', '#9CC3E0']
    , title='MAP@12 dos baselines em cada semana de validação'
)

fig.update_xaxes(title='')
fig.update_layout(legend_title='')
fig.show()

### O primeiro lugar é diferença real ou ruído?
Uma média maior não basta. Comparamos os dois primeiros colocados **nos mesmos clientes** da semana C: para cada cliente, a diferença de AP entre eles. Com dezenas de milhares de clientes, a média dessas diferenças tem um intervalo de 95% dado por `média ± 1,96 × desvio / √n`. Se o intervalo não cruza zero e a vantagem aparece nas três semanas, o primeiro colocado ganha de fato.

In [20]:
BASELINES = {
      'Popularidade geral': popularidade_geral
    , 'Popularidade da última semana': popularidade_ultima_semana
    , 'Recompra recente': recompra_recente
    , 'Popular na faixa etária': popular_faixa_etaria
}

primeiro, segundo = resultados.index[:2]

ap_primeiro = ap_por_cliente(BASELINES[primeiro](historico_c, clientes_c), semana_c)
ap_segundo = ap_por_cliente(BASELINES[segundo](historico_c, clientes_c), semana_c)

diferenca = ap_primeiro - ap_segundo
margem = 1.96 * diferenca.std() / np.sqrt(len(diferenca))

print(f'{primeiro} - {segundo}, semana C')
print(f'Diferença média de AP: {diferenca.mean():.4f}')
print(f'Intervalo de 95%: {diferenca.mean() - margem:.4f} a {diferenca.mean() + margem:.4f}')
print('Semanas em que o primeiro venceu:', (resultados.loc[primeiro, ['Semana A', 'Semana B', 'Semana C']] > resultados.loc[segundo, ['Semana A', 'Semana B', 'Semana C']]).sum(), 'de 3')

Recompra recente - Popular na faixa etária, semana C
Diferença média de AP: 0.0172
Intervalo de 95%: 0.0163 a 0.0181
Semanas em que o primeiro venceu: 3 de 3


### Onde estão os acertos
Quantos clientes receberam pelo menos um acerto do melhor baseline? O MAP@12 baixo costuma vir de muita gente com AP zero, não de acertos ruins.

In [21]:
faixas_ap = pd.cut(
      ap_primeiro
    , bins=[-0.001, 0, 0.1, 0.3, 1]
    , labels=['nenhum acerto', 'até 0,1', '0,1 a 0,3', 'acima de 0,3']
).value_counts(normalize=True).sort_index().reset_index()

faixas_ap.columns = ['AP do cliente', 'proporcao']

fig = px.bar(
      faixas_ap
    , x='AP do cliente'
    , y='proporcao'
    , text_auto='.1%'
    , color_discrete_sequence=['#3274A1']
    , title=f'Distribuição do AP por cliente — {primeiro}, semana C'
)

fig.update_yaxes(tickformat='.0%', title='Clientes')
fig.show()

### Próximos passos
- **Candidatos**: juntar as listas dos baselines (e outras fontes) em uma lista maior por cliente e medir o **recall**: que fração das compras reais está nela. Esse é o teto de qualquer modelo.
- **Atributos**: recência, frequência, preço, afinidade por `product_type_name` (a análise da etapa 1 vira atributo aqui), idade, canal — sempre calculados até o corte.
- **Modelos**: cada par cliente × candidato vira uma linha com alvo "comprou na semana? (0/1)". Logística, logística polinomial e LightGBM entram nesta mesma régua, e só ficam se superarem a tabela acima nas três semanas.